In [ ]:
import json
import requests
from pathlib import Path
from pprint import pprint

PRESIDIO_ANALYZER_URL = "http://localhost:3000/analyze"
PRESIDIO_ANON_URL = "http://localhost:5001/anonymize"

CLEAN_RESUME_PATH = Path("resume_cleaned.txt") 
MASKED_RESUME_PATH = Path("resume_masked.txt")

In [ ]:
if not CLEAN_RESUME_PATH.exists():
    raise FileNotFoundError(f"Cleaned resume not found at: {CLEAN_RESUME_PATH}")

with CLEAN_RESUME_PATH.open("r", encoding="utf-8") as f:
    clean_resume_text = f.read()

print("Loaded cleaned resume. First 500 chars:\n")
print(clean_resume_text[:500])


In [ ]:
def run_analyzer(text: str, entities=None, score_threshold: float | None = None):
    payload = {
        "text": text,
        "language": "en"
    }

    if entities or score_threshold is not None:
        payload["analyzer_config"] = {}
        if entities:
            payload["analyzer_config"]["entities"] = entities
        if score_threshold is not None:
            payload["analyzer_config"]["score_threshold"] = score_threshold

    print("\n[Analyzer] Sending request...")
    resp = requests.post(PRESIDIO_ANALYZER_URL, json=payload, timeout=20)
    resp.raise_for_status()
    results = resp.json()
    print(f"[Analyzer] Found {len(results)} entities.")
    return results


In [ ]:
def to_strict_analyzer_results(raw_results):
    strict = []
    for item in raw_results:
        strict.append({
            "start": int(item["start"]),
            "end": int(item["end"]),
            "entity_type": str(item["entity_type"]),
            "score": float(item.get("score", 0.85)),
        })
    return strict


In [ ]:
def run_anonymizer(text: str, analyzer_results, operators):
    payload = {
        "text": text,
        "analyzer_results": analyzer_results,
        "operators": operators
    }

    print("\n[Anonymizer] Sending request...")
    resp = requests.post(PRESIDIO_ANON_URL, json=payload, timeout=20)

    if resp.status_code == 422:
        print("\n[422 Schema Error] Raw response:")
        print(resp.text)
        raise ValueError("Anonymizer schema mismatch. Check operators block.")

    resp.raise_for_status()
    return resp.json()


In [ ]:
baseline_results = run_analyzer(
    clean_resume_text,
    entities=None,          # all entities (built-in + custom)
    score_threshold=0.35    # reasonable default
)

print("\n[Baseline Analyzer Results — first 10]:")
pprint(baseline_results[:10])


In [ ]:
test_texts = [
    "My LinkedIn is https://www.linkedin.com/in/yashgandhi",
    "GitHub: https://github.com/yash-ai",
    "Portfolio: https://yash.dev",
    "Random URL: https://example.com",
    "This is not a URL and should not be detected."
]

for i, t in enumerate(test_texts, start=1):
    print(f"\n=== Custom Recognizer Test #{i} ===")
    print("Text:", t)
    results = run_analyzer(
        t,
        entities=["URL", "LINKEDIN", "GITHUB", "PORTFOLIO"],  # adjust to your entity names
        score_threshold=0.0
    )
    pprint(results)


In [ ]:
fp_fn_cases = [
    ("Should detect LinkedIn", "linkedin.com/in/yashgandhi", True),
    ("Should detect GitHub", "github.com/yash-ai", True),
    ("Should detect portfolio", "yash.dev", True),
    ("Should NOT detect random URL as portfolio", "https://example.com", False),
    ("Should NOT detect plain text", "I like distributed systems and Rust.", False),
]

for label, text, should_detect in fp_fn_cases:
    print(f"\n=== {label} ===")
    print("Text:", text)
    results = run_analyzer(
        text,
        entities=["URL", "LINKEDIN", "GITHUB", "PORTFOLIO"],
        score_threshold=0.3
    )
    pprint(results)

    detected = len(results) > 0
    print("Detected:", detected, "| Expected detection:", should_detect)


In [ ]:
thresholds = [0.2, 0.35, 0.5, 0.7]

threshold_results_summary = {}

for th in thresholds:
    print(f"\n=== Threshold {th} on full resume ===")
    results = run_analyzer(
        clean_resume_text,
        entities=None,
        score_threshold=th
    )
    threshold_results_summary[th] = len(results)
    print(f"Entities found: {len(results)}")

print("\n[Threshold Summary]:")
for th, count in threshold_results_summary.items():
    print(f"  threshold={th}: {count} entities")


In [ ]:
strict_results = to_strict_analyzer_results(baseline_results)

strategies = {
    "mask_default": {
        "DEFAULT": {
            "operator_name": "mask",
            "type": "mask",
            "masking_char": "*",
            "chars_to_mask": 100
        }
    },
    "replace_token": {
        "DEFAULT": {
            "operator_name": "replace",
            "type": "replace",
            "new_value": "<PII>"
        }
    },
    "redact": {
        "DEFAULT": {
            "operator_name": "redact",
            "type": "redact",
            "new_value": ""
        }
    }
}




anonymized_variants = {}

for name, anonymizers in strategies.items():
    print(f"\n=== Strategy: {name} ===")
    try:
        data = run_anonymizer(clean_resume_text, strict_results, anonymizers)
        anonymized_variants[name] = data["text"]
        print("OK:", data["text"][:200])
    except Exception as e:
        print("FAILED:", e)



In [ ]:
final_strategy_name = "mask_default"   # or "replace_token" or "redact"

if final_strategy_name not in anonymized_variants:
    raise ValueError(f"Strategy '{final_strategy_name}' did not produce output. Check earlier errors.")

final_masked_text = anonymized_variants[final_strategy_name]

# --- MISSING WRITE STEP ADDED HERE ---
with MASKED_RESUME_PATH.open("w", encoding="utf-8") as f:
    f.write(final_masked_text)

print(f"🎉 Success! Masked resume written locally to: {MASKED_RESUME_PATH}")